# Lab 8: Introduction au Framework ADK et Multi-Provider

**Navigation** : [Index](../../README.md) | [Précédent <<](../../06-Agents-de-Donnees/Day3-Data-Agents/Labs/Lab7-Data-Analysis-Agent/Lab7-Data-Analysis-Agent.ipynb) | [Suivant >>](Lab9-First-ADK-Agent.ipynb)

## Objectifs d'apprentissage

A la fin de ce laboratoire, vous saurez :
1. Expliquer l'architecture du Google Agent Development Kit (ADK)
2. Configurer un environnement multi-provider (Gemini, vLLM, OpenAI)
3. Créer un premier client LLM avec votre provider choisi
4. Comparer les réponses de différents providers sur le même prompt

### Prérequis
- Python 3.10+
- Fichier `.env` configuré avec au moins un provider (voir section Configuration)
- Connaissance de base des LLMs (API, tokens, temperature)

### Durée estimée : 45-60 minutes

***

## Configuration requise

Avant de commencer, assurez-vous d'avoir configuré votre fichier `.env` :

```bash
# Exemple de configuration .env
ACTIVE_PROVIDER=gemini  # ou vllm, openai

# Gemini (optionnel)
GOOGLE_API_KEY=your_gemini_key

# vLLM (optionnel)
VLLM_BASE_URL=https://your-vllm-endpoint.com/v1
VLLM_MODEL=Qwen/Qwen2.5-72B-Instruct

# OpenAI (optionnel)
OPENAI_API_KEY=sk-...
```

***

## 1. Architecture Google ADK

Le **Agent Development Kit (ADK)** de Google est un framework pour construire des agents IA avec :

> **Repères bibliographiques.** Le concept d'agent IA à base de LLM (LLM-as-agent : perception → raisonnement → action via *tools*) est formalisé dans la synthèse de référence Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023. Le framework **Google ADK** (Agent Development Kit) en fournit une implémentation officielle (`google.github.io/adk-docs`, dépôt `google/adk-python`), et **LangChain** — lancé en octobre 2022 par H. Chase — popularise l'approche modulaire (composabilité de chaînes, *tools*, mémoire) sur laquelle s'appuie une large part de l'écosystème agent Python.

- **Agents** : Entités qui interagissent via des tools
- **Tools** : Fonctions que l'agent peut appeler
- **Sessions** : Gestion de l'état conversationnel
- **Memory** : Persistance du contexte

### Comparaison avec LangChain

| Aspect | ADK | LangChain |
|--------|-----|----------|
| **Philosophie** | Google-first, intégré GCP | Multi-provider natif |
| **Agents** | Agent classes avec tools | Runnable, chains, agents |
| **Mémoire** | Session state intégrée | Modules séparés |
| **Déploiement** | Vertex AI Agent Engine | Variable |
| **Providers** | Gemini natif, OpenAI compatible | 50+ providers |

## 2. Configuration de l'Environnement

### Installation des dépendances

In [1]:
# Installation des dependances (decommenter si necessaire)
# !pip install google-adk google-genai litellm pydantic-settings

Import des modules et vérification de l'installation.

In [2]:
import sys
import warnings
import os
from pathlib import Path

# Chemin vers 07-Orchestration-d-Agents depuis le répertoire de travail
track_dir = Path(os.getcwd()) / 'MyIA.AI.Notebooks' / 'ML' / 'DataScienceWithAgents' / '07-Orchestration-d-Agents'
sys.path.insert(0, str(track_dir))

warnings.filterwarnings('ignore', message=r'.*EXPERIMENTAL.*', category=UserWarning, module=r'google.adk')

from config import get_settings, get_provider_config
from utils.adk_runtime import build_agent, run_agent_turn

print('Modules importes avec succes')

Modules importes avec succes


### Vérification de la configuration

In [3]:
settings = get_settings()
config = get_provider_config(settings)

print(f'Provider actif: {config.provider.value}')
print(f'Modele: {config.model}')
print(f'Base URL: {config.base_url}')
api_display = '***' if config.api_key else 'Non requis (local)'
print(f'Cle API: {api_display}')

Provider actif: openrouter
Modele: openai/gpt-5.6-sol
Base URL: https://openrouter.ai/api/v1
Cle API: ***


Configuration detaillee du provider LLM.

In [4]:
print('Configuration complete :')
print(f'  Provider: {config.provider.value}')
print(f'  Model: {config.model}')
print(f'  API Key: {"Set" if config.api_key else "Not required"}')
print(f'  Base URL: {config.base_url}')

Configuration complete :
  Provider: openrouter
  Model: openai/gpt-5.6-sol
  API Key: Set
  Base URL: https://openrouter.ai/api/v1


## 3. Premier Test avec le Client LLM

Utilisons notre couche d'abstraction pour envoyer un prompt simple.

In [5]:
agent_intro = build_agent(
    name='lab8_intro',
    description='Agent d introduction ADK',
    instruction='Assistant expert ADK, reponds de maniere concise et precise.',
    config=config
)

response = await run_agent_turn(agent_intro, 'Explique en 2 phrases l architecture ADK.')

print('Reponse :')
print(response.response_text)

Reponse :
**Clarifying agent architecture**

I need to answer in French, and I should keep it to exactly two sentences. The mention of ADK seems ambiguous, likely referring to the Agent Development Kit architecture. I could explain the architecture in terms of hierarchical agents, runners, sessions, tools, and events while ensuring my explanation remains concise. This is a critical point to consider in my response.L’ADK structure une application autour d’agents autonomes ou hiérarchiques, chacun doté d’un modèle, d’instructions et d’outils. Un *Runner* orchestre leur exécution et les échanges d’événements, tandis que les services de session, mémoire et artefacts gèrent l’état et les données.


Test de génération simple avec le client configuré.

In [6]:
response2 = await run_agent_turn(
    agent_intro,
    'Quels sont les 3 composants principaux d un agent ADK ?'
)

print('Composants ADK :')
print(response2.response_text)

Composants ADK :
**Clarifying ADK Components**

The user needs the answer in French. I'm thinking ADK stands for Google Agent Development Kit, which probably involves three main components: the model, instructions, and tools. In terms of the ADK agent's anatomy, I might mention its identity, including the name and description. I can explain that the model is about reasoning, instructions govern behavior, and tools are for actions. I want to keep it concise and clear for the user!Les 3 composants principaux d’un agent ADK sont :

1. **Le modèle (Model)** : le LLM qui raisonne et génère les réponses.
2. **Les instructions (Instructions)** : elles définissent son rôle, son comportement et ses objectifs.
3. **Les outils (Tools)** : les fonctions ou services externes qu’il peut utiliser pour agir ou récupérer des informations.


### Lecture du résultat : la réponse décrit un agent générique, pas ADK

La question portait sur ADK ; la réponse énumère **Capteurs**, **Processeur**, **Actionneurs**. C'est la boucle *perception → raisonnement → action* des agents fondés sur un LLM (référence 1, Xi et al.), transposée au vocabulaire d'un système robotique — **pas** le modèle objet du framework. Les composants que la section 1 de ce laboratoire documente sont autres : **Agents**, **Tools**, **Sessions**, **Memory** (référence 2, documentation officielle d'ADK).

La cellule de test précédente porte le même défaut sous une forme plus discrète : interrogée sur l'architecture d'ADK, elle répond « une structure modulaire » sans nommer un seul composant du framework. Le modèle répond donc de mémoire, et cette mémoire le ramène au vocabulaire générique des agents.

**Ce qu'il faut en retenir** : une réponse de LLM n'est pas une source. Sur un framework récent, la sortie d'un agent se confronte à la documentation du framework — ici la section 1 et la référence 2 de ce notebook. La sortie de la cellule ci-dessus est conservée telle quelle : un désaccord visible entre la prose et la sortie est un support pédagogique, une sortie réécrite à la main n'en serait plus une.


### Test avec prompt système

In [7]:
agent_system = build_agent(
    name='lab8_system',
    description='Agent avec instruction systeme',
    instruction='Tu es un expert en IA. Explique les concepts de maniere claire.',
    config=config
)

response3 = await run_agent_turn(
    agent_system,
    'Explique le concept de Session dans ADK.'
)

print('Session dans ADK :')
print(response3.response_text)

Session dans ADK :
Dans **ADK (Agent Development Kit)**, une **Session** représente une interaction continue entre un utilisateur et un système d’agents. Elle sert de conteneur pour le contexte d’une conversation.

## Ce que contient une Session

Une session est généralement identifiée par :

- `app_name` : l’application concernée ;
- `user_id` : l’utilisateur ;
- `session_id` : l’instance précise de conversation ;
- `events` : l’historique des messages, réponses et actions ;
- `state` : des données structurées nécessaires pendant l’échange ;
- `last_update_time` : la date de dernière modification.

Par exemple, deux conversations distinctes d’un même utilisateur correspondent normalement à deux sessions différentes.

## Session, historique et état

ADK distingue notamment :

### 1. Les événements

Les **Events** décrivent ce qui s’est passé pendant la session :

- message utilisateur ;
- réponse de l’agent ;
- appel d’un outil ;
- résultat d’un outil ;
- transfert entre agents ;
- mod

## 4. Providers disponibles et test sur le provider actif

Énumérons les providers disponibles, puis testons le même prompt sur le provider actif (`ACTIVE_PROVIDER` du `.env`). La comparaison croisée de plusieurs providers sur un même prompt est l'objet de l'exercice 1 ci-dessous.

In [8]:
from config.providers import ProviderType

print('Providers disponibles:')
for p in ProviderType:
    print(f'  - {p.value}')

r_mp = await run_agent_turn(agent_intro, 'Quelle est la difference entre un LLM et un agent ?')
print('Reponse:', r_mp.response_text[:200], '...')

Providers disponibles:
  - gemini
  - openai
  - openrouter
  - qwen
  - vllm
  - lmstudio


Reponse: **Clarifying LLMs and Agents**

I need to provide a concise and precise answer in French. I want to explain the difference between a Large Language Model (LLM) and an agent, possibly in the context of ...


### Interprétation

La réponse ci-dessus provient du provider configuré dans `.env` (`ACTIVE_PROVIDER`). Pour les sorties committées de ce notebook, cette configuration effective était **`openrouter`** avec le modèle **`openai/gpt-4.1-mini`** (visible dans les sorties des cellules de configuration ci-dessus) — et non le « Qwen réel » annoncé par la spec #13925, rédigée avant vérification de l'environnement d'exécution (traçage demandé par #14547).

Pour tester un autre provider, modifiez votre fichier `.env` et redémarrez le kernel, ou instanciez un client avec une configuration explicite :

```python
from config import ProviderConfig, ProviderType

# Exemple pour vLLM
vllm_config = ProviderConfig(
    provider=ProviderType.VLLM,
    model="Qwen/Qwen2.5-72B-Instruct",
    base_url="https://your-vllm-endpoint.com/v1",
    api_key=None
)
vllm_client = LLMClient(vllm_config)
```

## 5. Interface de Chat avec Historique

Le client supporte également une interface de chat avec historique des messages.

In [9]:
sid = 'lab8_chat_session'

r_a = await run_agent_turn(agent_intro, 'Bonjour, je suis nouveau avec ADK.', session_id=sid)
r_b = await run_agent_turn(agent_intro, 'Peux-tu me rappeler ce que j ai dit ?', session_id=sid)
print('Tour 1:', r_a.response_text)
print('Tour 2:', r_b.response_text)

Tour 1: **Crafting a French response**

I realize I need to provide an answer in French. It might be a good idea to start with a welcome and then explain what I'm discussing. I suspect "ADK" stands for Agent Development Kit, given its context with developers, but I have to be concise in my response. I want to make sure I convey the right information clearly and effectively!Bonjour et bienvenue ! 👋

ADK (**Agent Development Kit**) permet de créer, orchestrer et tester des agents IA. Pour débuter, vous pouvez apprendre à :

1. installer ADK ;
2. créer un agent simple avec un nom, des instructions et un modèle ;
3. lui ajouter des outils ;
4. le tester localement ;
5. le déployer.

Souhaitez-vous commencer par l’installation ou par un exemple d’agent minimal ?
Tour 2: Vous avez dit : « Peux-tu me rappeler ce que j’ai dit ? »


## 6. Architecture des Frameworks DS-STAR et MLE-STAR

DS-STAR et MLE-STAR sont deux agents de référence (State-of-the-Art) conçus par Google Research pour la data science et l'ingénierie ML — ils incarnent le paradigme d'agent LLM *planner-coder* (décomposition de tâche → génération de code → exécution → raffinement) décrit par Xi et al. (2025) et appliqué à la compétition Kaggle et au cycle d'expérimentation ML.

Ce track 07-Orchestration-d-Agents intègre les frameworks de recherche Google :

### DS-STAR (Data Science Agent)

Architecture Planner-Coder-Verifier pour la data science autonome :

```mermaid
flowchart TD
    FA["File Analyzer"] --> P["Planner"]
    P --> C["Coder"]
    P --> V["Verifier"]
    C --> E["Executor"]
    E --> V
```

Le planificateur distribue le travail entre génération et vérification, tandis que l'exécuteur renvoie les résultats au vérificateur pour fermer la boucle de raffinement.

**Performance** : 45.2% accuracy sur DABStep benchmark

### MLE-STAR (ML Engineering Agent)

Extension avec recherche web et optimisation automatique :

- Web Search pour modèles SOTA
- Ablation studies ciblées
- Ensemble stratégies automatisées

**Performance** : 63.6% médailles sur MLE-Bench-Lite

## Exercice : Comparaison Multi-Provider

Maintenant que vous avez compris l'architecture, testez votre capacité à utiliser différents providers pour la même tâche.

In [10]:
# Exercice : Comparaison Multi-Provider
# Objectif : Creer une fonction qui compare les reponses de differents agents/ Providers sur le meme prompt
# TODO: Definissez une liste de prompts de test pertinents pour la data science
# Indice : utilisez des questions sur l'ADK, les agents, ou la data science
test_prompts = None  # Exemple: ['Explique l'architecture ADK.', 'A quoi sert un agent ?']

# TODO: Implementez une fonction qui execute chaque prompt sur plusieurs agents/providers
# Utilisez build_agent pour creer des agents avec differentes configurations
# puis run_agent_turn pour executer chaque prompt
# Indice: mesurez le temps avec time.time()
async def compare_providers(prompts, config):
    """
    Compare les reponses de differents agents pour une liste de prompts.
        Args:
        prompts: Liste de prompts a tester
        config: Configuration ADK
            Returns:
        dict: Resultats par prompt avec reponses et temps
    """
    results = {}
    # TODO: Implementez la boucle de test
    # Indice: creez des agents avec build_agent, puis appelez run_agent_turn
    # for prompt in prompts:
    #     agent = build_agent(name='test', instruction='Reponds de maniere concise.', config=config)
    #     start = time.time()
    #     resp = await run_agent_turn(agent, prompt)
    #     results[prompt] = {'response': resp.response_text, 'time': time.time() - start}
    return results

# TODO: Executez la comparaison et affichez les resultats
# results = await compare_providers(test_prompts, config)
# for prompt, data in results.items():
#     print(f"Prompt: {prompt[:50]}...")
#     print(f"Reponse: {data['response'][:100]}...")
#     print(f"Temps: {data['time']:.2f}s")
print("Exercice a completer : comparaison multi-provider avec ADK")

Exercice a completer : comparaison multi-provider avec ADK


## Exercice : Chat Multi-Tours avec Contexte Data Science

Utilisez l'interface `chat()` du client LLM pour construire une conversation spécialisée en data science. L'objectif est de simuler un assistant qui guide un utilisateur dans son analyse de données, étape par étape.

### Objectifs
1. Construire un historique de conversation avec 3 échanges
2. Utiliser un system prompt spécialisé data science
3. Observer comment le contexte précédent influence les réponses

**Indice :**
- Utilisez `client.chat(messages)` avec une liste de dictionnaires `{"rôle": "user"/"assistant", "content": "..."}`
- Commencez par un message system via le premier élément de la liste

In [11]:
# Exercice : Chat Multi-Tours avec Contexte Data Science
# Objectif : Simuler un assistant d'analyse de donnees en 3 echanges
# TODO: Construisez un agent specialise en data science
# Utilisez build_agent avec une instruction appropriee
ds_agent = None  # Exemple: build_agent(name='ds_expert', instruction='Tu es un expert en data science.', config=config)

# TODO: Definissez un identifiant de session pour maintenir le contexte
ds_sid = None  # Exemple: 'ds_session_001'

# TODO: Premier echange - l'utilisateur decrit son dataset
# Indice: utilisez run_agent_turn avec session_id pour maintenir l'historique
# question_1 = "J'ai un dataset de ventes avec date, produit, region, quantite et prix."
# resp_1 = await run_agent_turn(ds_agent, question_1, session_id=ds_sid)

# TODO: Deuxieme echange - l'utilisateur demande une analyse specifique
# question_2 = "Quelles visualisations me recommandes-tu pour comparer les regions ?"
# resp_2 = await run_agent_turn(ds_agent, question_2, session_id=ds_sid)

# TODO: Troisieme echange - approfondissement
# question_3 = "Comment detecter des valeurs aberrantes dans les prix ?"
# resp_3 = await run_agent_turn(ds_agent, question_3, session_id=ds_sid)

# TODO: Affichez les 3 reponses et observez la progression du contexte
# print("=== Echange 1 ===")
# print(resp_1.response_text)
# print("=== Echange 2 ===")
# print(resp_2.response_text)
# print("=== Echange 3 ===")
# print(resp_3.response_text)
print("Exercice a completer : chat multi-tours avec agent data science via ADK")

Exercice a completer : chat multi-tours avec agent data science via ADK


## Exercice : Exploration des Paramètres de Generation

Expérimentez avec les paramètres `temperature` et `max_tokens` pour comprendre leur impact sur la qualité des réponses d'un agent. L'objectif est de trouver les paramètres optimaux pour différentes tâches d'agent.

### Objectifs
1. Tester 3 valeurs de température (0.1, 0.7, 1.5) sur un prompt technique
2. Observer l'impact de `max_tokens` sur la longueur des réponses
3. Déterminer les paramètres idéaux pour du code generation vs. du texte créatif

**Indice :**
- `client.generate(prompt, temperature=0.1, max_tokens=100)` pour contrôler la génération
- Température basse = réponses déterministes (idéal pour du code)
- Température haute = réponses créatives (idéal pour du brainstorming)

In [12]:
# Exercice : Exploration des parametres de generation
# Objectif : Trouver les parametres optimaux selon la tache de l'agent
# TODO: Definissez des prompts pour differentes taches
prompt_technique = "Explique la difference entre un agent base sur des tools et un agent base sur du code execution."  # Ne pas modifier
prompt_code = "Ecris une fonction Python qui calcule la moyenne mobile d'une serie temporelle."  # Ne pas modifier

# TODO: Testez differentes temperatures et comparez les reponses
# Utilisez build_agent avec differentes configurations de temperature
# puis run_agent_turn pour executer les prompts
# Indice: essayez temperature=0.1, 0.7, 1.5
temperatures = [0.1, 0.7, 1.5]
results_temp = {}

# TODO: Implementez la boucle de test pour les temperatures
# for t in temperatures:
#     agent = build_agent(name='temp_test', instruction='Reponds de maniere detaillee.', config=config, temperature=t)
#     resp = await run_agent_turn(agent, prompt_technique)
#     results_temp[t] = resp.response_text
#     print(f"--- Temperature {t} ---")
#     print(results_temp[t][:200])
#     print()

# TODO: Testez l'impact de max_tokens sur un prompt de code
# Indice: essayez max_tokens=50 vs max_tokens=500
# agent_short = build_agent(name='short', instruction='Ecris du code concis.', config=config, max_tokens=50)
# agent_long = build_agent(name='long', instruction='Ecris du code complet.', config=config, max_tokens=500)
# resp_short = await run_agent_turn(agent_short, prompt_code)
# resp_long = await run_agent_turn(agent_long, prompt_code)
# print(f"Short (50 tokens): {resp_short.response_text[:100]}...")
# print(f"Long (500 tokens): {resp_long.response_text[:200]}...")

# TODO: Resumez vos conclusions dans un dictionnaire
# Adaptez les valeurs en fonction de vos observations
parametres_ideaux = {
    "code_generation": {"temperature": None, "max_tokens": None},  # Remplacez None
    "analyse_creative": {"temperature": None, "max_tokens": None},  # Remplacez None
    "brainstorming": {"temperature": None, "max_tokens": None}   # Remplacez None
}
print("Exercice a completer : exploration des parametres temperature et max_tokens avec ADK")

Exercice a completer : exploration des parametres temperature et max_tokens avec ADK


## Résumé et Prochaines Étapes

### Ce que nous avons appris

1. **Configuration multi-provider** : Un seul fichier `.env` permet de switcher entre Gemini, vLLM, OpenAI
2. **Abstraction LiteLLM** : Interface unifiée pour tous les providers
3. **Client LLM simple** : `generate()` et `chat()` pour interagir avec n'importe quel modèle
4. **Architecture DS-STAR** : Framework Planner-Coder-Verifier pour la data science autonome

### Points clés à retenir

| Concept | Description |
|---------|-------------|
| `ProviderConfig` | Configuration d'un provider LLM |
| `LLMClient` | Client unifié pour tous les providers |
| `generate()` | Génération simple avec prompt |
| `chat()` | Conversation multi-tours avec historique |

### Prochaines étapes

- **Lab 9** : Créer un premier agent ADK avec tools Python pour analyser des DataFrames
- **Lab 10** : Implémenter le File Analyzer de DS-STAR
- **Lab 11** : Boucle Planner-Coder-Verifier

***

**Navigation** : [Index](../../README.md) | [Précédent <<](../../06-Agents-de-Donnees/Day3-Data-Agents/Labs/Lab7-Data-Analysis-Agent/Lab7-Data-Analysis-Agent.ipynb) | [Suivant >> Lab 9 - First ADK Agent](Lab9-First-ADK-Agent.ipynb)

## Ressources

- [Google ADK Documentation](https://google.github.io/adk-docs)
- [Google ADK Samples (dépôt d'exemples)](https://github.com/google/adk-samples)
- [DS-STAR Paper](https://research.google/blog/ds-star-a-state-of-the-art-versatile-data-science-agent/)
- [MLE-STAR Paper](https://research.google/blog/mle-star-a-state-of-the-art-machine-learning-engineering-agents/)
- [LiteLLM Documentation](https://docs.litellm.ai/)

## Références

1. Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023. Synthèse de référence sur les agents IA à base de LLM (architecture perception-raisonnement-action, *tools*, mémoire, cadres planner-coder).
2. Google, *Agent Development Kit (ADK)*, documentation officielle, `google.github.io/adk-docs` (dépôt `google/adk-python`). Framework agent-first de Google (sessions, mémoire, Vertex AI Agent Engine).
3. H. Chase, *LangChain*, octobre 2022, `langchain.com` / `github.com/langchain-ai/langchain`. Framework modulaire open-source pour applications LLM (chaînes composables, *tools*, mémoire) — référence de l'écosystème agent Python.
4. Google Research, *DS-STAR: A State-of-the-Art Versatile Data Science Agent*, 2025, `research.google/blog/ds-star-a-state-of-the-art-versatile-data-science-agent/`. Agent SOTA data science (paradigme planner-coder).
5. Google Research, *MLE-STAR: A State-of-the-Art Machine Learning Engineering Agent*, 2025, `research.google/blog/mle-star-a-state-of-the-art-machine-learning-engineering-agents/`. Agent SOTA ingénierie ML (cycle d'expérimentation, Kaggle).